In [1]:
import os
import requests
from collections import Counter
from google.transit import gtfs_realtime_pb2
import pandas as pd

FUENTES = {
    "ACES": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-ace",
    "BDFMS":  "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-bdfm",
    "G":  "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-g",
    "JZ": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-jz",
    "NQRW": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-nqrw",
    "L": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-l",
    "1234567S": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs",
    "SIR":  "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-si",
}
HEADERS =  {} # API_KEY si la tuviera

In [2]:
def obtener_dataframe(url):
    response = requests.get(url, headers=HEADERS, timeout=30)

    if response.status_code != 200:
        print(f"Error {response.status_code}")
        print(response.text[:200])
        return None

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    registros = []

    for entity in feed.entity:
        if entity.HasField("trip_update"):
            tu = entity.trip_update

            for stop in tu.stop_time_update:
                registro = {
                    "trip_id": tu.trip.trip_id,
                    "route_id": tu.trip.route_id,
                    "start_date": tu.trip.start_date if tu.trip.HasField("start_date") else None,
                    "start_time": tu.trip.start_time if tu.trip.HasField("start_time") else None,
                    "stop_id": stop.stop_id,
                    "arrival_time": (
                        pd.to_datetime(stop.arrival.time, unit="s", utc=True)
                        if stop.HasField("arrival") else None
                    ),
                    "departure_time": (
                        pd.to_datetime(stop.departure.time, unit="s", utc=True)
                        if stop.HasField("departure") else None
                    ),
                }

                registros.append(registro)

    return pd.DataFrame(registros)

In [3]:
def inspeccionar_endpoint(nombre, url):
    print(f"\n==================== {nombre} ====================")
    df = obtener_dataframe(url)

    if df is None or df.empty:
        print("DataFrame vacío")
        return

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nFilas:", len(df))

    print("\n% Nulos:")
    print((df.isna().mean() * 100).round(2))

    print("\nEjemplo:")
    print(df.head(3))

    return df

In [6]:
dfs = []

for nombre, url in FUENTES.items():
    df = inspeccionar_endpoint(nombre, url)
    if df is None or df.empty:
        continue
    df = df.copy()
    df["endpoint"] = nombre   # o feed_name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


==================== ACES ====================

Columnas:
['trip_id', 'route_id', 'start_date', 'start_time', 'stop_id', 'arrival_time', 'departure_time']

Tipos:
trip_id                        object
route_id                       object
start_date                     object
start_time                     object
stop_id                        object
arrival_time      datetime64[ns, UTC]
departure_time    datetime64[ns, UTC]
dtype: object

Filas: 1483

% Nulos:
trip_id           0.0
route_id          0.0
start_date        0.0
start_time        0.0
stop_id           0.0
arrival_time      0.0
departure_time    0.0
dtype: float64

Ejemplo:
          trip_id route_id start_date start_time stop_id  \
0  028200_A..N09R        A   20260216   04:42:40    A11N   
1  028200_A..N09R        A   20260216   04:42:40    A10N   
2  028200_A..N09R        A   20260216   04:42:40    A09N   

               arrival_time            departure_time  
0 2026-02-16 11:22:17+00:00 2026-02-16 11:22:17+00:00  
1

In [7]:
df_all

,trip_id,route_id,start_date,start_time,stop_id,arrival_time,departure_time,endpoint
0,028200_A..N09R,A,20260216,04:42:40,A11N,2026-02-16 11:22:17+00:00,2026-02-16 11:22:17+00:00,ACES
1,028200_A..N09R,A,20260216,04:42:40,A10N,2026-02-16 11:23:34+00:00,2026-02-16 11:23:34+00:00,ACES
2,028200_A..N09R,A,20260216,04:42:40,A09N,2026-02-16 11:26:19+00:00,2026-02-16 11:26:19+00:00,ACES
3,028200_A..N09R,A,20260216,04:42:40,A07N,2026-02-16 11:29:34+00:00,2026-02-16 11:29:34+00:00,ACES
4,028200_A..N09R,A,20260216,04:42:40,A06N,2026-02-16 11:30:34+00:00,2026-02-16 11:30:34+00:00,ACES
...,...,...,...,...,...,...,...,...
8534,042600_SI..S,SI,20260216,07:06:00,S15S,2026-02-16 12:44:00+00:00,2026-02-16 12:44:00+00:00,SIR
8535,042600_SI..S,SI,20260216,07:06:00,S14S,2026-02-16 12:46:00+00:00,2026-02-16 12:46:00+00:00,SIR
8536,042600_SI..S,SI,20260216,07:06:00,S13S,2026-02-16 12:48:00+00:00,2026-02-16 12:48:00+00:00,SIR
8537,042600_SI..S,SI,20260216,07:06:00,S11S,2026-02-16 12:50:00+00:00,2026-02-16 12:50:00+00:00,SIR
